In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 09 · Inferencia — del modelo exportado a una estimación que cualquiera puede pedir

Etapa 9 (`especificaciones/09-app.md`, `specs/007-app-inference/`). Un modelo entrenado no es un producto: un producto
recibe los datos de una persona real, a veces incompletos, y responde con algo útil o con un **mensaje que dice qué
corregir**. Este notebook muestra ese puente con los mismos pasos que hace la app:

1. Cargar `models/model.joblib` y `models/input_schema.json` (exportados en la etapa 8). **Sin Databricks.**
2. Agregar al esquema los **nombres de los municipios**, para que la app muestre «MEDELLÍN (5001)» y no solo el código.
3. Estimar el puntaje de **4 perfiles inventados** (no salen de los datos), escritos con los textos del formulario.
4. Ver un **ejemplo rechazado**.

**No se abre la prueba.** La exportación desde el registro y la comprobación de que el exportado es el evaluado ya se
hicieron en la etapa 8 (`reports/08_registro.md`). La lógica está en `src/proyecto/inference/`; aquí solo se orquesta.

## 0. Modelo y esquema

In [2]:
import json

import pandas as pd

from proyecto.inference.form import SIN_ESTRATO, SIN_RESPUESTA, form_to_row, reference_range
from proyecto.inference.predict import InputValidationError, load_model, predict_score
from proyecto.inference.schema import load_schema, save_schema

raiz = Path.cwd().resolve().parent
schema_path = raiz / "models" / "input_schema.json"
modelo = load_model(raiz / "models" / "model.joblib")
schema = load_schema(schema_path)
assert schema["task"] == "regression"
print(f"modelo: {type(modelo).__name__} con pasos {[nombre for nombre, _ in modelo.steps]}")
print(f"esquema: {len(schema['numeric'])} numéricas + {len(schema['categorical'])} categóricas; "
      f"{len(schema['nullable'])} admiten «{SIN_RESPUESTA}»")

modelo: Pipeline con pasos ['prep', 'model']
esquema: 5 numéricas + 16 categóricas; 12 admiten «Sin respuesta»


## 1. Nombres de los municipios para la app

El modelo usa los **códigos** DANE de municipio. Para que la app muestre el nombre, se agrega al esquema la clave
`municipality_names` (código → nombre), calculada **solo con el entrenamiento** (`data/processed/train.joblib`), que
trae el nombre junto al código. Son nombres de municipios, no datos de personas. Solo se agrega esa clave: el resto del
esquema de la etapa 8 queda igual (se comprueba).

In [3]:
import joblib

train = joblib.load(raiz / "data" / "processed" / "train.joblib")
pares = [("cole_cod_mcpio_ubicacion", "cole_mcpio_ubicacion"), ("estu_cod_reside_mcpio", "estu_mcpio_reside")]
nombres = pd.concat(
    [train[[codigo, nombre]].dropna().set_axis(["codigo", "nombre"], axis=1) for codigo, nombre in pares]
).drop_duplicates()
nombres["codigo"] = nombres["codigo"].astype(int).astype(str)   # mismo formato que las categorías del esquema
assert not nombres["codigo"].duplicated().any(), "hay códigos con más de un nombre"

conocidos = set(schema["categories"]["cole_cod_mcpio_ubicacion"]) | set(schema["categories"]["estu_cod_reside_mcpio"])
municipality_names = {c: n for c, n in sorted(zip(nombres["codigo"], nombres["nombre"])) if c in conocidos}
assert set(municipality_names) == conocidos, "algún código del esquema no tiene nombre"

antes = {k: v for k, v in schema.items() if k != "municipality_names"}
schema["municipality_names"] = municipality_names
save_schema(schema, schema_path)
despues = load_schema(schema_path)
assert {k: v for k, v in despues.items() if k != "municipality_names"} == antes, "el resto del esquema cambió"
print(f"{len(municipality_names)} municipios con nombre; por ejemplo 5001 → {municipality_names['5001']}")
del train

139 municipios con nombre; por ejemplo 5001 → MEDELLÍN


## 2. Cuatro perfiles inventados

Escritos con los mismos textos que elige una persona en la app («3 a 4», «Tres», «Sin estrato»). `form_to_row` los
traduce a las 21 columnas del modelo y `predict_score` valida y estima. El puntaje se muestra redondeado, con la
referencia «estimación ± 33» (el error típico, MAE 33,3 en la prueba), que **no** es un intervalo de confianza.

In [4]:
base = {
    "cole_cod_mcpio_ubicacion": "5001", "cole_naturaleza": "OFICIAL", "cole_area_ubicacion": "URBANO",
    "cole_jornada": "MAÑANA", "cole_caracter": "ACADÉMICO", "cole_genero": "MIXTO", "cole_bilingue": "N",
    "cole_sede_principal": "S", "edad": 17, "estu_genero": "F", "estu_cod_reside_mcpio": "5001", "estrato": "2",
    "fami_personashogar": "3 a 4", "fami_cuartoshogar": "Tres", "fami_educacionmadre": "Secundaria (Bachillerato) completa",
    "fami_educacionpadre": "Primaria completa", "fami_tieneautomovil": "No", "fami_tienecomputador": "Si",
    "fami_tieneinternet": "Si", "fami_tienelavadora": "Si",
}
perfiles = {
    "A · oficial urbano en Medellín, estrato 2": base,
    "B · no oficial bilingüe en Envigado, estrato 5": {
        **base, "cole_cod_mcpio_ubicacion": "5266", "estu_cod_reside_mcpio": "5266", "cole_naturaleza": "NO OFICIAL",
        "cole_jornada": "COMPLETA", "cole_bilingue": "S", "edad": 16, "estu_genero": "M", "estrato": "5",
        "fami_cuartoshogar": "Cuatro", "fami_educacionmadre": "Postgrado",
        "fami_educacionpadre": "Educación profesional completa", "fami_tieneautomovil": "Si",
    },
    "C · oficial rural en Turbo, estrato 1, sin computador ni internet": {
        **base, "cole_cod_mcpio_ubicacion": "5837", "estu_cod_reside_mcpio": "5837", "cole_area_ubicacion": "RURAL",
        "cole_jornada": "UNICA", "cole_caracter": "TÉCNICO", "edad": 18, "estrato": "1",
        "fami_personashogar": "5 a 6", "fami_cuartoshogar": "Dos", "fami_educacionmadre": "Primaria incompleta",
        "fami_educacionpadre": "Primaria incompleta", "fami_tienecomputador": "No", "fami_tieneinternet": "No",
        "fami_tienelavadora": "No",
    },
    "D · incompleto: Sin estrato y varios «Sin respuesta» (Rionegro)": {
        **base, "cole_cod_mcpio_ubicacion": "5615", "estu_cod_reside_mcpio": "5615", "cole_jornada": "TARDE",
        "edad": SIN_RESPUESTA, "estu_genero": "M", "estrato": SIN_ESTRATO, "fami_personashogar": SIN_RESPUESTA,
        "fami_cuartoshogar": SIN_RESPUESTA, "fami_educacionmadre": SIN_RESPUESTA, "fami_educacionpadre": "No sabe",
        "fami_tieneautomovil": SIN_RESPUESTA,
    },
}

filas = []
for nombre, respuestas in perfiles.items():
    resultado = predict_score(form_to_row(respuestas, schema), modelo, schema)
    puntaje = float(resultado["puntaje_estimado"].iloc[0])
    bajo, alto = reference_range(puntaje)
    filas.append({"perfil": nombre, "puntaje estimado": round(puntaje), "referencia (± 33)": f"{bajo} a {alto}",
                  "advertencias": len(resultado.attrs["warnings"])})
pd.DataFrame(filas).set_index("perfil")

,puntaje estimado,referencia (± 33),advertencias
perfil,,,
"A · oficial urbano en Medellín, estrato 2",250,217 a 283,0
"B · no oficial bilingüe en Envigado, estrato 5",323,290 a 356,0
"C · oficial rural en Turbo, estrato 1, sin computador ni internet",212,179 a 245,0
D · incompleto: Sin estrato y varios «Sin respuesta» (Rionegro),247,214 a 280,0


## 3. Un ejemplo rechazado

Una edad imposible (12 años: nadie presenta Saber 11 a esa edad, decisión 4 del EDA) y un dato obligatorio sin elegir.
Lo que se espera: **ningún puntaje** y un mensaje que dice qué corregir.

In [5]:
rechazado = {**base, "edad": 12, "cole_jornada": None}
try:
    predict_score(form_to_row(rechazado, schema), modelo, schema)
    print("ERROR: debía rechazarse")
except InputValidationError as error:
    print("Rechazado, sin puntaje. Mensaje:")
    print(error)

Rechazado, sin puntaje. Mensaje:
Edad (años cumplidos): escribe un número entero entre 13 y 80, o marca «Sin respuesta». Jornada del colegio: elige una opción.


## Qué queda de esta etapa

- La app (`app/streamlit_app.py`) usa exactamente estas funciones: `streamlit run app/streamlit_app.py` desde la raíz.
- Pruebas: `tests/test_app.py` (formulario y app con AppTest) y `tests/test_inference.py`.
- Despliegue: `reports/09_despliegue.md` (en esta etapa no se desplegó).